# Phase 10 — Deep Learning for PHM (readiness gate)

## First: deep learning comes after a valid baseline

Phase 9 deliberately did not fit or compare RUL models. Phase 8 found no cohort-wide verified EoL labels, and the MAT date text conflicts with `timeEpoch` interpreted as Unix seconds. Consequently, there is no justified RUL target and no working supervised baseline to improve. **We will not train an LSTM, GRU, or other neural network in this phase.**

This lesson teaches the data representation and architecture choices. It uses actual IGBT transient-feature captures to demonstrate chronological sequence windows and train-only normalization, but creates no RUL labels and trains no model. The output is an input tensor, not a PHM predictor. Deep learning becomes appropriate only after targets, time alignment, baseline performance, independent validation, and adequate device/run counts are established.

## 1. From measurements to sequences

A **sequence** is an ordered list of feature vectors from one device/run:

$$X = [x_1, x_2, \ldots, x_T], \qquad x_t \in \mathbb{R}^{F}$$

Here, `T` is the number of observations and `F` is the number of features per observation. This project has one candidate-feature row per transient capture. It is not a uniformly sampled continuous waveform: captures are separated in recorded time, sampling interval `dt` can vary within the capture, and the stress protocol changes. Preserve device identity and source-file segments; do not join unrelated files into an artificial continuous sequence.

A **sliding window** of length `L` creates overlapping input examples such as `[x_1…x_L]`, `[x_2…x_(L+1)]`, etc. The stride controls how far the window moves. If `stride=1`, adjacent windows share almost all their observations, so they are highly dependent. A chronological train/test boundary must be chosen before making windows, and no window may use future values unavailable at prediction time.

## 2. Inspect capture files and create input windows only

We use a small set of Phase 5 descriptors as a shape demonstration: collector-emitter voltage RMS and peak-to-peak, plus gate-command peak-to-peak as context. Windows are built **within each source file** so a `Device4` to `Device4b` boundary is never silently bridged. The `date` field gives a chronological ordering for this local demonstration; its clock synchronization and alignment to exposure are not validated, so do not interpret it as a verified aging coordinate.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd()
if not (project_root / "data" / "features").is_dir():
    project_root = project_root.parent

feature_path = project_root / "data" / "features" / "transient_candidate_features.csv"
if not feature_path.is_file():
    raise FileNotFoundError(
        f"{feature_path} was not found. Run notebooks/03_feature_engineering.ipynb first."
    )

captures = pd.read_csv(feature_path)
feature_columns = [
    "collectorEmitterVoltage_rms",
    "collectorEmitterVoltage_peak_to_peak",
    "gateSignalVoltage_peak_to_peak",
]
required = {"device_folder", "source_file", "date", *feature_columns}
missing = sorted(required.difference(captures.columns))
if missing:
    raise ValueError(f"Feature table is missing required columns: {missing}")

captures["capture_time"] = pd.to_datetime(
    captures["date"], format="%m/%d/%Y %I:%M:%S %p.%f", errors="raise"
)
captures["is_check_file"] = captures["source_file"].str.contains(
    "check", case=False, regex=False
)
aging_captures = captures.loc[~captures["is_check_file"]].copy()
if aging_captures.empty:
    raise ValueError("No non-check capture rows are available.")

segment_sizes = aging_captures.groupby(
    ["device_folder", "source_file"], observed=True
).size().sort_values(ascending=False)
segment_sizes.head(10)

## 3. Split first, normalize second, window third

Neural networks are sensitive to feature scale, so normalization (for example, z-scoring) is often helpful. The scaler must be fitted **only on training observations**. Fitting it on the full run leaks information about the future distribution into training. Apply the training mean and standard deviation unchanged to later data; never normalize each test window using its own future values if that would be unavailable in deployment.

The following is an input-preparation demonstration on the longest non-`check` source-file segment: first 70% of that file for training, final 30% for a chronological illustration. This is not a valid model evaluation (one segment is not independent validation), and no target is available. Windows are formed separately on each side of the boundary so they cannot straddle it. A deployed forecast could use historical training-period context to warm up the first test window, but that policy must be defined explicitly.

In [ ]:
def make_windows(values: np.ndarray, lookback: int, stride: int = 1) -> np.ndarray:
    """Return input-only sliding windows with shape (n_windows, lookback, n_features)."""
    values = np.asarray(values, dtype=float)
    if values.ndim != 2:
        raise ValueError("values must have shape (n_observations, n_features)")
    if lookback < 1 or stride < 1:
        raise ValueError("lookback and stride must be positive integers")
    if values.shape[0] < lookback:
        return np.empty((0, lookback, values.shape[1]), dtype=float)
    starts = range(0, values.shape[0] - lookback + 1, stride)
    return np.stack([values[start:start + lookback] for start in starts])

device_name, source_file = segment_sizes.index[0]
segment = aging_captures.loc[
    (aging_captures["device_folder"] == device_name)
    & (aging_captures["source_file"] == source_file)
].sort_values("capture_time").reset_index(drop=True)

split_at = int(np.floor(0.70 * len(segment)))
if split_at < 12 or len(segment) - split_at < 12:
    raise ValueError("Selected segment is too short for a 12-capture window on both sides.")

scaler = StandardScaler()
train_raw = segment.loc[:split_at - 1, feature_columns].to_numpy(dtype=float)
test_raw = segment.loc[split_at:, feature_columns].to_numpy(dtype=float)
if not np.isfinite(train_raw).all() or not np.isfinite(test_raw).all():
    raise ValueError("Candidate features contain non-finite values; resolve them before scaling.")

train_scaled = scaler.fit_transform(train_raw)
test_scaled = scaler.transform(test_raw)
lookback = 12
X_train = make_windows(train_scaled, lookback=lookback, stride=1)
X_test = make_windows(test_scaled, lookback=lookback, stride=1)

print(f"Demonstration segment: {device_name} / {source_file}")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")
print(f"Raw train / test rows: {train_raw.shape} / {test_raw.shape}")
print(f"Window length: {lookback}; X_train shape: {X_train.shape}; X_test shape: {X_test.shape}")
print("Scaler means learned from training rows only:", scaler.mean_)
print("Test arrays were transformed with the training scaler; no labels or model were created.")

## 4. Sequence-to-one prediction

For **sequence-to-one**, the model receives a window with shape `(lookback, features)` and returns one scalar (or one vector) for a specified prediction time. In RUL PHM, the intended mapping might be:

$$[x_{t-L+1}, \ldots, x_t] \longrightarrow \widehat{\mathrm{RUL}}_t$$

The target must correspond to the window endpoint, use a justified EoL and consistent unit, and not leak future observations into any input. A different task—forecasting the next feature value—would instead map the window to `x_(t+1)` and is **not RUL prediction**. This dataset has no reliable RUL targets, so `X_train`/`X_test` above have no `y_train`/`y_test`; we intentionally do not invent them from the last observation.

## 5. Why LSTM or GRU may help—and why not yet

### LSTM (Long Short-Term Memory)

An LSTM is a recurrent neural network with gated updates and a cell state. The gates regulate what information is retained, updated, or exposed, helping model dependencies across longer sequences than a basic recurrent network. In PHM, this can be useful when degradation evolves gradually and its current state depends on a history of operating conditions and measurements. It can learn nonlinear temporal patterns without hand-crafted lag features.

That flexibility comes with more parameters, tuning choices, and data needs. LSTM memory does not automatically mean physical damage memory; it can learn device identity, stress schedule, file artifacts, or sensor drift. It does not define EoL, handle censoring automatically, or guarantee extrapolation beyond seen trajectories.

### GRU (Gated Recurrent Unit)

A GRU uses update/reset gates and combines the cell/hidden state rather than maintaining the separate LSTM cell state. It is often a simpler, faster recurrent alternative with fewer parameters. It may be preferable when data are limited and a gated sequence model is justified, but it still needs representative independent runs, leakage-safe validation, and a valid target. There is no universal rule that GRU beats LSTM or vice versa.

For either architecture, compare against simpler feature-based baselines, tune only inside training folds, report uncertainty across held-out devices, and inspect predictions by device and operating stage. With four devices and an unverified RUL target, training either network now would mainly risk memorizing the experiment.

## 6. Later architectures—only if the problem calls for them

- **1D CNN:** filters detect local motifs over a sequence, such as short changes or repeated patterns. It can be efficient and parallelizable; for raw switching waveforms it may learn edges/ringing, but sampling interval, event alignment, acquisition bandwidth, and calibration must be handled first. A CNN over capture-level features detects local temporal patterns, not within-capture waveform structure.
- **CNN-LSTM:** a CNN extracts local patterns and an LSTM models their order/longer context. This hybrid is useful when both local motifs and longer temporal evolution matter, but adds complexity and needs enough independent device trajectories to validate each part.
- **Transformer:** self-attention relates distant sequence positions and allows parallel sequence processing. It can help with long, rich, multivariate histories and large diverse datasets, but commonly needs substantially more data and careful positional/time-gap encoding. Attention does not solve timestamp errors or confer physical interpretability. **Do not start with a Transformer** for four short, irregularly sampled device runs.

Architecture selection follows the signal and question: local waveform motifs suggest CNNs; sequential memory suggests LSTM/GRU; both may motivate CNN-LSTM; long-range relationships at adequate scale may motivate Transformers. Always let data volume, sampling, targets, and validation constrain complexity.

## 7. Leakage traps specific to sequence learning

1. Randomly split overlapping windows: nearly identical inputs can appear in train and test.
2. Create windows across device, test, or file boundaries: the model sees an artificial transition and may learn identity/provenance.
3. Fit the normalizer on the full trajectory or use per-window future statistics: test/future distributions leak into training.
4. Randomly split captures from a single run when the goal is future forecasting: future context contaminates evaluation.
5. Attach a label computed from a future event to a feature window while also including post-event or future-derived values in that window.

Define the deployment question first. For future prediction on known devices, reserve later time blocks. For unseen-device generalization, hold out whole devices; with only four devices, performance uncertainty will remain large. If overlapping windows are used, split the raw timeline/run first, then create windows within each partition, or use a carefully defined warm-up context without sharing target outcomes.

## Phase 10 learning check

1. What are the `(batch, time, features)` dimensions of an LSTM/GRU input, and what does a sliding window represent?
2. For RUL sequence-to-one prediction, which timestamp does the single target belong to?
3. Why must the scaler be fitted on training data only?
4. What temporal role might an LSTM/GRU model that a tabular baseline misses?
5. When might a CNN, CNN-LSTM, or Transformer be useful—and what data would each need?
6. Why are the feature windows here only a representation demo, not a trained PHM result?

**Readiness decision:** no deep model is trained. First resolve the Phase 8 event-time/EoL issues, then establish a valid Phase 9 baseline and device-aware evaluation, and only then test whether sequence learning adds value.